# BasementDrugDiscovery
## Notebook 03 -- Prepare Receptor Structures for Docking

**What this notebook does:**
Downloads CYP51 crystal structures from the Protein Data Bank for the fungal targets and the human counter-screen. Cleans each structure by removing water and the original co-crystallized ligand, keeps the heme cofactor, adds hydrogens, and converts to PDBQT format for AutoDock Vina. Defines a docking grid box for each receptor based on the position of the original ligand.

**Targets:**
- *Candida albicans* CYP51
- *Aspergillus fumigatus* CYP51B
- *Cryptococcus neoformans* CYP51
- Human CYP51 (counter-screen for selectivity)

**Input:** PDB IDs entered via widget, no hardcoded paths

**Output:** Cleaned receptor PDBQT files, grid box parameters saved as JSON, one record per target

---

### Cell 1 -- Load all required tools

In [1]:
import os
import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

from rdkit import Chem, rdBase

import Bio
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.PDBIO import Select as PDBSelect

import ipywidgets as widgets
from IPython.display import display, clear_output

import subprocess

print('All tools loaded successfully.')
print(f'RDKit version: {rdBase.rdkitVersion}')
print(f'Biopython version: {Bio.__version__}')

All tools loaded successfully.
RDKit version: 2026.03.3
Biopython version: 1.87


### Cell 2 -- Set output directory

Choose where receptor files and structure data will be saved.

In [2]:
output_dir_widget = widgets.Text(
    value=str(Path.home() / 'BasementDrugDiscovery' / 'data' / 'structures'),
    description='Receptor output folder:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)

confirm_btn = widgets.Button(
    description='Confirm and create directory',
    button_style='primary',
    layout=widgets.Layout(width='280px')
)

output_status = widgets.Output()

def confirm_output(b):
    with output_status:
        clear_output()
        p = Path(output_dir_widget.value)
        p.mkdir(parents=True, exist_ok=True)
        print(f'Output directory ready: {p}')

confirm_btn.on_click(confirm_output)
display(output_dir_widget)
display(confirm_btn)
display(output_status)

Text(value='/home/sardism/BasementDrugDiscovery/data/structures', description='Receptor output folder:', layou…

Button(button_style='primary', description='Confirm and create directory', layout=Layout(width='280px'), style…

Output()

### Cell 3 -- Enter target PDB IDs

Type the PDB ID for each target structure. Each entry needs a short label that will be used to name the output files.

Leave a field blank if you do not yet have a structure for that target -- it will be skipped.

In [4]:
# Step 1 -- Choose how many targets to set up
num_targets_widget = widgets.BoundedIntText(
    value=3,
    min=1,
    max=10,
    description='Number of targets:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

generate_btn = widgets.Button(
    description='Generate target fields',
    button_style='primary',
    layout=widgets.Layout(width='250px')
)

target_fields_output = widgets.Output()
target_widgets = {}

def generate_target_fields(b):
    global target_widgets
    target_widgets = {}
    
    with target_fields_output:
        clear_output()
        n = num_targets_widget.value
        print(f'Enter details for {n} targets:\n')
        
        for i in range(1, n + 1):
            label = widgets.Label(
                value=f'--- Target {i} ---',
                layout=widgets.Layout(width='500px')
            )
            
            name_w = widgets.Text(
                value='',
                description='Target name:',
                placeholder='e.g. candida, aspergillus, human',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='500px')
            )
            
            pdb_w = widgets.Text(
                value='',
                description='PDB ID:',
                placeholder='e.g. 5TZ1',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='500px')
            )
            
            chain_w = widgets.Text(
                value='A',
                description='Chain to keep:',
                placeholder='e.g. A',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='500px')
            )
            
            display(label)
            display(name_w)
            display(pdb_w)
            display(chain_w)
            print()
            
            target_widgets[i] = {
                'name': name_w,
                'pdb_id': pdb_w,
                'chain': chain_w
            }

generate_btn.on_click(generate_target_fields)

print('How many receptor structures do you want to prepare?')
display(num_targets_widget)
display(generate_btn)
display(target_fields_output)
print('\nFill in all fields, then run the next cell.')

How many receptor structures do you want to prepare?


BoundedIntText(value=3, description='Number of targets:', layout=Layout(width='300px'), max=10, min=1, style=D…

Button(button_style='primary', description='Generate target fields', layout=Layout(width='250px'), style=Butto…

Output()


Fill in all fields, then run the next cell.


### Cell 4 -- Download structures from the Protein Data Bank

Downloads the raw PDB file for each entered target. No account needed, the PDB is fully open.

In [6]:
def download_pdb_structure(pdb_id, output_dir):
    """
    Download a structure file from the RCSB Protein Data Bank.
    Returns the path to the saved file, or None if download failed.
    """
    pdb_id = pdb_id.strip().upper()
    if not pdb_id:
        return None
    
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    output_path = Path(output_dir) / f'{pdb_id}_raw.pdb'
    
    try:
        urllib.request.urlretrieve(url, output_path)
        print(f'Downloaded {pdb_id}: {output_path}')
        return output_path
    except Exception as e:
        print(f'Failed to download {pdb_id}: {e}')
        return None


downloaded_structures = {}

for target_name, widget in target_widgets.items():
    # Handle both old format (widget directly) and new format (dict of widgets)
    if hasattr(widget, 'value'):
        pdb_id = widget.value.strip()
        chain = 'A'
    else:
        pdb_id = widget['pdb_id'].value.strip()
        chain = widget['chain'].value.strip()
        target_name = widget['name'].value.strip().lower().replace(' ', '_') or target_name

    if not pdb_id:
        print(f'Skipping {target_name} -- no PDB ID entered.')
        continue
    
    print(f'\nDownloading {target_name} ({pdb_id})...')
    path = download_pdb_structure(pdb_id, output_dir_widget.value)
    if path:
        downloaded_structures[target_name] = {
            'pdb_id': pdb_id.upper(),
            'chain': chain,
            'raw_path': str(path)
        }

print(f'\nTotal structures downloaded: {len(downloaded_structures)}')
print(list(downloaded_structures.keys()))

Downloaded 5FRB: /home/sardism/BasementDrugDiscovery/data/structures/5FRB_raw.pdb



Downloaded 5V5Z: /home/sardism/BasementDrugDiscovery/data/structures/5V5Z_raw.pdb



Downloaded 6UEZ: /home/sardism/BasementDrugDiscovery/data/structures/6UEZ_raw.pdb

Total structures downloaded: 3
['aspergillus_fumigatus', 'candida_albicans_sc5314', 'homo_sapiens']


### Cell 5 -- Inspect each structure

Before cleaning, look at what is actually in each file. This shows chains, ligands, and water count so we know what we are removing.

In [7]:
def inspect_structure(pdb_path):
    """
    Print a summary of chains, het groups, and water molecules in a PDB file.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('s', pdb_path)
    
    chains = set()
    het_groups = {}
    water_count = 0
    
    for model in structure:
        for chain in model:
            chains.add(chain.id)
            for residue in chain:
                het_flag = residue.id[0]
                if het_flag == 'W':
                    water_count += 1
                elif het_flag.startswith('H_'):
                    resname = residue.get_resname()
                    het_groups[resname] = het_groups.get(resname, 0) + 1
        break  # only first model
    
    print(f'  Chains: {sorted(chains)}')
    print(f'  Water molecules: {water_count}')
    print(f'  Heteroatom groups: {het_groups}')
    
    return chains, het_groups, water_count


structure_info = {}

for target_name, info in downloaded_structures.items():
    print(f'\n{target_name} ({info["pdb_id"]}):')
    chains, het_groups, water_count = inspect_structure(info['raw_path'])
    structure_info[target_name] = {
        'chains': sorted(chains),
        'het_groups': het_groups,
        'water_count': water_count
    }


aspergillus_fumigatus (5FRB):
  Chains: ['A']
  Water molecules: 0
  Heteroatom groups: {'HEM': 1, 'VT2': 1}

candida_albicans_sc5314 (5V5Z):
  Chains: ['A']
  Water molecules: 0
  Heteroatom groups: {'HEM': 1, '1YN': 1}

homo_sapiens (6UEZ):
  Chains: ['A', 'B']
  Water molecules: 514
  Heteroatom groups: {'HEM': 2, 'LAN': 2}


### Cell 6 -- Select the chain and ligand to use for each target

Many of these structures contain two copies of the protein in the same file. We need exactly one chain per receptor, and we need to identify which heteroatom group is the co-crystallized inhibitor, since that defines where the binding site grid box goes. The heme cofactor (HEM) should always be kept with the protein, the inhibitor is what we use only to define the grid box, then it gets removed before docking.

In [8]:
chain_widgets = {}
ligand_widgets = {}

print('For each target, confirm which chain to keep and which heteroatom group')
print('is the co-crystallized ligand (used to define the binding site).')
print('HEM (heme) will always be kept automatically -- do not enter it as the ligand.\n')

for target_name, info in structure_info.items():
    print(f'--- {target_name} ---')
    print(f'Available chains: {info["chains"]}')
    print(f'Heteroatom groups found: {info["het_groups"]}')
    
    chain_widgets[target_name] = widgets.Text(
        value=info['chains'][0] if info['chains'] else 'A',
        description=f'{target_name} chain to keep:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    non_heme_groups = [k for k in info['het_groups'].keys() if k not in ('HEM', 'HOH')]
    default_ligand = non_heme_groups[0] if non_heme_groups else ''
    
    ligand_widgets[target_name] = widgets.Text(
        value=default_ligand,
        description=f'{target_name} ligand code (for grid box):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    display(chain_widgets[target_name])
    display(ligand_widgets[target_name])
    print()

For each target, confirm which chain to keep and which heteroatom group
is the co-crystallized ligand (used to define the binding site).
HEM (heme) will always be kept automatically -- do not enter it as the ligand.

--- aspergillus_fumigatus ---
Available chains: ['A']
Heteroatom groups found: {'HEM': 1, 'VT2': 1}


Text(value='A', description='aspergillus_fumigatus chain to keep:', layout=Layout(width='400px'), style=TextSt…

Text(value='VT2', description='aspergillus_fumigatus ligand code (for grid box):', layout=Layout(width='400px'…


--- candida_albicans_sc5314 ---
Available chains: ['A']
Heteroatom groups found: {'HEM': 1, '1YN': 1}


Text(value='A', description='candida_albicans_sc5314 chain to keep:', layout=Layout(width='400px'), style=Text…

Text(value='1YN', description='candida_albicans_sc5314 ligand code (for grid box):', layout=Layout(width='400p…


--- homo_sapiens ---
Available chains: ['A', 'B']
Heteroatom groups found: {'HEM': 2, 'LAN': 2}


Text(value='A', description='homo_sapiens chain to keep:', layout=Layout(width='400px'), style=TextStyle(descr…

Text(value='LAN', description='homo_sapiens ligand code (for grid box):', layout=Layout(width='400px'), style=…

### Cell 7 -- Clean each structure

Keep only the selected chain, the protein atoms, and the heme cofactor. Remove water and the co-crystallized ligand (its coordinates are saved separately first, to define the grid box). Save as a clean PDB file.

In [9]:
class ReceptorSelect(PDBSelect):
    """
    Biopython selector that keeps only the chosen chain,
    standard protein residues, and the heme cofactor.
    Removes water and the co-crystallized ligand.
    """
    def __init__(self, chain_id):
        self.chain_id = chain_id
    
    def accept_chain(self, chain):
        return chain.id == self.chain_id
    
    def accept_residue(self, residue):
        het_flag = residue.id[0]
        resname = residue.get_resname()
        
        # Standard amino acid
        if het_flag == ' ':
            return True
        # Keep heme cofactor
        if resname == 'HEM':
            return True
        # Remove everything else -- water, ligand, other hetero groups
        return False


class LigandSelect(PDBSelect):
    """
    Biopython selector that extracts only the co-crystallized ligand
    from the chosen chain, used to determine grid box coordinates.
    """
    def __init__(self, chain_id, ligand_resname):
        self.chain_id = chain_id
        self.ligand_resname = ligand_resname
    
    def accept_chain(self, chain):
        return chain.id == self.chain_id
    
    def accept_residue(self, residue):
        return residue.get_resname() == self.ligand_resname


def clean_structure(raw_path, chain_id, ligand_resname, output_dir, target_name):
    """
    Clean a structure file -- extract protein+heme and the ligand separately.
    Returns paths to the cleaned receptor and the extracted ligand.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('s', raw_path)
    
    io = PDBIO()
    io.set_structure(structure)
    
    # Save cleaned receptor (protein + heme only)
    receptor_path = Path(output_dir) / f'{target_name}_receptor.pdb'
    io.save(str(receptor_path), ReceptorSelect(chain_id))
    
    # Save the ligand separately if specified
    ligand_path = None
    if ligand_resname:
        io.set_structure(structure)
        ligand_path = Path(output_dir) / f'{target_name}_ligand.pdb'
        io.save(str(ligand_path), LigandSelect(chain_id, ligand_resname))
    
    return receptor_path, ligand_path


cleaned_structures = {}

for target_name, info in downloaded_structures.items():
    chain_id = chain_widgets[target_name].value.strip()
    ligand_resname = ligand_widgets[target_name].value.strip()
    
    print(f'Cleaning {target_name} -- chain {chain_id}, ligand {ligand_resname or "none"}...')
    
    receptor_path, ligand_path = clean_structure(
        info['raw_path'], chain_id, ligand_resname,
        output_dir_widget.value, target_name
    )
    
    cleaned_structures[target_name] = {
        'pdb_id': info['pdb_id'],
        'chain': chain_id,
        'ligand_resname': ligand_resname,
        'receptor_pdb': str(receptor_path),
        'ligand_pdb': str(ligand_path) if ligand_path else None
    }
    print(f'  Receptor saved: {receptor_path}')
    if ligand_path:
        print(f'  Ligand saved: {ligand_path}')

print(f'\nAll structures cleaned: {len(cleaned_structures)}')

Cleaning aspergillus_fumigatus -- chain A, ligand VT2...


  Receptor saved: /home/sardism/BasementDrugDiscovery/data/structures/aspergillus_fumigatus_receptor.pdb
  Ligand saved: /home/sardism/BasementDrugDiscovery/data/structures/aspergillus_fumigatus_ligand.pdb
Cleaning candida_albicans_sc5314 -- chain A, ligand 1YN...
  Receptor saved: /home/sardism/BasementDrugDiscovery/data/structures/candida_albicans_sc5314_receptor.pdb
  Ligand saved: /home/sardism/BasementDrugDiscovery/data/structures/candida_albicans_sc5314_ligand.pdb
Cleaning homo_sapiens -- chain A, ligand LAN...


  Receptor saved: /home/sardism/BasementDrugDiscovery/data/structures/homo_sapiens_receptor.pdb
  Ligand saved: /home/sardism/BasementDrugDiscovery/data/structures/homo_sapiens_ligand.pdb

All structures cleaned: 3


### Cell 8 -- Calculate the docking grid box for each target

The grid box defines the 3D region AutoDock Vina searches for binding poses. We center it on the co-crystallized ligand with generous padding to allow some flexibility in pose search.

In [10]:
GRID_PADDING = 10.0  # Angstroms of padding around the ligand on each side

def calculate_grid_box(ligand_pdb_path, padding=GRID_PADDING):
    """
    Calculate the center and size of a docking grid box
    based on the coordinates of the co-crystallized ligand.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('ligand', ligand_pdb_path)
    
    coords = []
    for atom in structure.get_atoms():
        coords.append(atom.get_coord())
    
    if not coords:
        return None
    
    coords = np.array(coords)
    
    min_coords = coords.min(axis=0)
    max_coords = coords.max(axis=0)
    center = (min_coords + max_coords) / 2
    size = (max_coords - min_coords) + (2 * padding)
    
    return {
        'center_x': round(float(center[0]), 3),
        'center_y': round(float(center[1]), 3),
        'center_z': round(float(center[2]), 3),
        'size_x': round(float(size[0]), 3),
        'size_y': round(float(size[1]), 3),
        'size_z': round(float(size[2]), 3),
        'padding_used': padding
    }


for target_name, info in cleaned_structures.items():
    if info['ligand_pdb'] is None:
        print(f'{target_name}: no ligand specified, skipping grid box calculation.')
        print(f'  You will need to set the grid box manually for this target.')
        continue
    
    grid = calculate_grid_box(info['ligand_pdb'])
    cleaned_structures[target_name]['grid_box'] = grid
    
    print(f'{target_name} grid box:')
    print(f'  Center: ({grid["center_x"]}, {grid["center_y"]}, {grid["center_z"]})')
    print(f'  Size:   ({grid["size_x"]}, {grid["size_y"]}, {grid["size_z"]})')

aspergillus_fumigatus grid box:
  Center: (300.648, 10.702, 11.379)
  Size:   (43.922, 25.546, 24.354)
candida_albicans_sc5314 grid box:
  Center: (-38.038, -19.281, 24.569)
  Size:   (34.002, 38.327, 34.71)
homo_sapiens grid box:
  Center: (-29.059, -33.0, 15.889)
  Size:   (25.533, 33.077, 30.719)


### Cell 9 -- Convert receptors to PDBQT using OpenBabel

AutoDock Vina requires receptors in PDBQT format. We use system OpenBabel for this conversion, calling it as a subprocess. Hydrogens are added at this step.

In [11]:
def convert_receptor_to_pdbqt(receptor_pdb_path, output_dir, target_name):
    """
    Convert a cleaned receptor PDB file to PDBQT using OpenBabel.
    Adds hydrogens and assigns partial charges.
    """
    pdbqt_path = Path(output_dir) / f'{target_name}_receptor.pdbqt'
    
    cmd = [
        'obabel',
        str(receptor_pdb_path),
        '-O', str(pdbqt_path),
        '-xr',       # rigid receptor flag
        # Gasteiger charges fail silently (0 molecules converted, empty output)
        # on structures containing the HEM cofactor -- the Gasteiger charge
        # model has no parameters for the heme iron, so OpenBabel aborts the
        # whole write. EEM charges handle Fe correctly and give equivalent
        # per-atom partial charges for the standard amino acids.
        '--partialcharge', 'eem'
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    # obabel returns exit code 0 even when it converts 0 molecules, so an
    # empty output file is the real signal of failure -- exists() alone
    # isn't enough, since obabel creates (truncates) the file before it
    # decides whether it can actually write a molecule to it.
    if pdbqt_path.exists() and pdbqt_path.stat().st_size > 0:
        return str(pdbqt_path), True, None
    else:
        error = result.stderr.strip() or 'obabel produced an empty PDBQT file'
        return None, False, error


for target_name, info in cleaned_structures.items():
    print(f'Converting {target_name} to PDBQT...')
    
    pdbqt_path, success, error = convert_receptor_to_pdbqt(
        info['receptor_pdb'], output_dir_widget.value, target_name
    )
    
    if success:
        cleaned_structures[target_name]['receptor_pdbqt'] = pdbqt_path
        print(f'  Success: {pdbqt_path}')
    else:
        cleaned_structures[target_name]['receptor_pdbqt'] = None
        print(f'  Failed: {error}')

print('\nReceptor conversion complete.')

Converting aspergillus_fumigatus to PDBQT...


  Success: /home/sardism/BasementDrugDiscovery/data/structures/aspergillus_fumigatus_receptor.pdbqt
Converting candida_albicans_sc5314 to PDBQT...


  Success: /home/sardism/BasementDrugDiscovery/data/structures/candida_albicans_sc5314_receptor.pdbqt
Converting homo_sapiens to PDBQT...


  Success: /home/sardism/BasementDrugDiscovery/data/structures/homo_sapiens_receptor.pdbqt

Receptor conversion complete.


### Cell 10 -- Save the target configuration

Save all receptor paths and grid box parameters to a JSON file. This becomes the input for the docking notebook -- one file describing every target completely.

In [12]:
config_path = Path(output_dir_widget.value) / 'docking_targets.json'

with open(config_path, 'w') as f:
    json.dump(cleaned_structures, f, indent=2)

print(f'Target configuration saved: {config_path}')
print('\nSummary of all prepared targets:')
print('=' * 60)

for target_name, info in cleaned_structures.items():
    print(f'\n{target_name.upper()} ({info["pdb_id"]})')
    print(f'  Chain used: {info["chain"]}')
    print(f'  Ligand for grid box: {info["ligand_resname"] or "none specified"}')
    print(f'  Receptor PDBQT: {"ready" if info.get("receptor_pdbqt") else "FAILED"}')
    if 'grid_box' in info:
        g = info['grid_box']
        print(f'  Grid center: ({g["center_x"]}, {g["center_y"]}, {g["center_z"]})')
        print(f'  Grid size: ({g["size_x"]}, {g["size_y"]}, {g["size_z"]})')
    else:
        print(f'  Grid box: NOT SET -- needs manual configuration')

print('\n' + '=' * 60)
ready_count = sum(1 for info in cleaned_structures.values() if info.get('receptor_pdbqt') and 'grid_box' in info)
print(f'Targets fully ready for docking: {ready_count} of {len(cleaned_structures)}')

Target configuration saved: /home/sardism/BasementDrugDiscovery/data/structures/docking_targets.json

Summary of all prepared targets:

ASPERGILLUS_FUMIGATUS (5FRB)
  Chain used: A
  Ligand for grid box: VT2
  Receptor PDBQT: ready
  Grid center: (300.648, 10.702, 11.379)
  Grid size: (43.922, 25.546, 24.354)

CANDIDA_ALBICANS_SC5314 (5V5Z)
  Chain used: A
  Ligand for grid box: 1YN
  Receptor PDBQT: ready
  Grid center: (-38.038, -19.281, 24.569)
  Grid size: (34.002, 38.327, 34.71)

HOMO_SAPIENS (6UEZ)
  Chain used: A
  Ligand for grid box: LAN
  Receptor PDBQT: ready
  Grid center: (-29.059, -33.0, 15.889)
  Grid size: (25.533, 33.077, 30.719)

Targets fully ready for docking: 3 of 3


---
## What comes next

The receptor structures are prepared and the docking targets are defined. Each receptor has a clean PDBQT file and a grid box centered on the original inhibitor binding site.

**Notebook 04** runs AutoDock Vina, screening all 2,864 prepared ligands against every target in this configuration file.

**GitHub:** https://github.com/sardism/BasementDrugDiscovery

**Note:** All results are computational predictions requiring experimental validation.